# Record & Label — single-notebook BC data collection

**셀 실행 순서:**
1. Config / Imports / Calib / Motor / Controls — 한 번만 실행
2. **Camera + Threads** — 카메라 문제 생길 때마다 재실행
3. **Recording** — 세션마다 재실행
4. Cleanup — 끝낼 때

**Output**: `filename xpos ypos segment steer_tel speed_tel`

## Part A — Record

In [1]:
# --- Config ---
from pathlib import Path
SESSION_NAME = "session01"
DATA_ROOT    = Path.home() / "rover_data"
FPS          = 15
CAM_W, CAM_H = 1280, 720
ENABLE_MOTOR = True
UART_PORT    = "/dev/ttyUSB0"
UART_BAUD    = 115200
CALIB_PATH   = Path.home() / "team/main/ros2_ws/src/rover_stereo/config/stereo_calib.yaml"

In [2]:
# --- Imports ---
import sys, os, time, threading
sys.path.insert(0, str(Path.home() / "team"))

import numpy as np
import cv2
import yaml
import ipywidgets as widgets
from IPython.display import display

# Explicit package paths so we get the vendored copies (with the sync=false
# jetcam fix), not the originals under ~/HYU-ECL3003/rover/.
from calibration.camera import Camera

In [3]:
# --- Calibration ---
c = yaml.safe_load(open(CALIB_PATH))
K1, D1 = np.array(c["K1"]), np.array(c["D1"])
K2, D2 = np.array(c["K2"]), np.array(c["D2"])
R, T   = np.array(c["R"]),  np.array(c["T"])
W, H   = c["image_size"]

R1, R2, P1, P2, Q, roi1, roi2 = cv2.stereoRectify(K1, D1, K2, D2, (W, H), R, T, alpha=0.0)
m1x, m1y = cv2.initUndistortRectifyMap(K1, D1, R1, P1, (W, H), cv2.CV_16SC2)
m2x, m2y = cv2.initUndistortRectifyMap(K2, D2, R2, P2, (W, H), cv2.CV_16SC2)

x1, y1, w1, h1 = roi1
x2, y2, w2, h2 = roi2
xa, ya = max(x1, x2), max(y1, y2)
xb, yb = min(x1 + w1, x2 + w2), min(y1 + h1, y2 + h2)
ROI = (xa, ya, xb - xa, yb - ya) if (xb > xa and yb > ya) else (0, 0, W, H)
print("rectify ROI:", ROI, "  fx:", round(float(P1[0, 0]), 2))

rectify ROI: (0, 0, 1280, 720)   fx: 1288.22


In [4]:
# --- Motor ---
motor = None
if ENABLE_MOTOR:
    from control.base_ctrl import BaseController
    motor = BaseController(UART_PORT, UART_BAUD)
    print("motor ready on", UART_PORT)
else:
    print("ENABLE_MOTOR=False")

motor ready on /dev/ttyUSB0


In [5]:
# --- Shared state + widget controls ---
state = dict(steer=0.0, speed=0.0, segment="common", recording=False, stop=False)

STEER_STEP = 0.80
SPEED_STEP = 0.05

def update_state(key):
    if   key == "up":
        state["speed"]  = max(-1.0, state["speed"]  - SPEED_STEP)
    elif key == "down":
        state["speed"]  = min( 1.0, state["speed"]  + SPEED_STEP)
    elif key == "left":
        state["steer"]  = max(-1.0, state["steer"]  - STEER_STEP)
        state["speed"]  = max(-1.0, state["speed"]  - SPEED_STEP * 2)
    elif key == "right":
        state["steer"]  = min( 1.0, state["steer"]  + STEER_STEP)
        state["speed"]  = max(-1.0, state["speed"]  - SPEED_STEP * 2)
    elif key == "straight":
        state["steer"]  = 0.0
        state["speed"]  = min( 1.0, state["speed"]  + SPEED_STEP * 2)
    elif key == "space":
        state["steer"]  = 0.0; state["speed"] = 0.0
    elif key == "1":     state["segment"] = "common"
    elif key == "2":     state["segment"] = "left"
    elif key == "3":     state["segment"] = "right"
    elif key == "p":     state["segment"] = "pause"
    elif key == "r":     state["recording"] = not state["recording"]
    elif key == "esc":   state["stop"] = True

btn = lambda desc, key, style="": widgets.Button(
    description=desc, button_style=style, layout=widgets.Layout(width="80px", height="40px")
)

b_up       = btn("▲ fwd",    "up")
b_down     = btn("▼ bwd",    "down")
b_left     = btn("◀ L",      "left")
b_right    = btn("▶ R",      "right")
b_space    = btn("■ stop",   "space",    "warning")
b_straight = btn("straight", "straight")
b_rec      = btn("⏺ rec",    "r",        "danger")
b_esc      = btn("✕ end",    "esc",      "danger")
b_seg1     = btn("common",   "1")
b_seg2     = btn("left",     "2")
b_seg3     = btn("right",    "3")
b_pause    = btn("pause",    "p")

for b, k in [(b_up,"up"),(b_down,"down"),(b_left,"left"),(b_right,"right"),
             (b_space,"space"),(b_straight,"straight"),
             (b_rec,"r"),(b_esc,"esc"),
             (b_seg1,"1"),(b_seg2,"2"),(b_seg3,"3"),(b_pause,"p")]:
    b.on_click(lambda _, key=k: update_state(key))

dpad = widgets.VBox([b_up, widgets.HBox([b_left, b_space, b_right]), b_down, b_straight])
ctrl = widgets.VBox([widgets.HBox([b_rec, b_esc]),
                     widgets.HBox([b_seg1, b_seg2, b_seg3, b_pause])])
display(widgets.HBox([dpad, ctrl]))
print("widget controls ready")

widget controls ready


### Camera + Threads (카메라 문제 생기면 이 셀만 재실행)

In [22]:
# --- Camera open + flush + threads (한 셀에 통합) ---

# 1. 이전 스레드 정리
state["stop"] = True
time.sleep(0.3)

# 2. 이전 카메라 정리
for name in ["cam_l", "cam_r"]:
    cam = globals().get(name)
    if cam is not None:
        try: cam.stop()
        except: pass
        try: cam._cam.cap.release()
        except: pass
time.sleep(0.3)

# 3. 카메라 open
cam_l = Camera(sensor_id=0, capture_width=CAM_W, capture_height=CAM_H, capture_fps=FPS*2)
cam_r = Camera(sensor_id=1, capture_width=CAM_W, capture_height=CAM_H, capture_fps=FPS*2)
print("cameras up:", cam_l.running(), cam_r.running())

# 4. 누적된 프레임 시간 기반으로 flush (open과 사용 시점 사이 누적분 제거)
def flush_cam(cap, duration=0.5):
    deadline = time.time() + duration
    n = 0
    while time.time() < deadline:
        cap.grab(); n += 1
    return n

nl = flush_cam(cam_l._cam.cap)
nr = flush_cam(cam_r._cam.cap)
print(f"flushed L={nl} R={nr} frames")

# 5. state 초기화 + 스레드 시작
state.update(stop=False, speed=0.0, steer=0.0, recording=False)
latest = {"L": None, "R": None}

def cam_thread_l():
    while not state["stop"]:
        try:
            l = cam_l.read()
            if l is not None: latest["L"] = l
        except: time.sleep(0.01)

def cam_thread_r():
    while not state["stop"]:
        try:
            r = cam_r.read()
            if r is not None: latest["R"] = r
        except: time.sleep(0.01)

threading.Thread(target=cam_thread_l, daemon=True).start()
threading.Thread(target=cam_thread_r, daemon=True).start()
time.sleep(0.3)
print("camera threads ready")

GST_ARGUS: Cleaning up


CONSUMER: Done Success
GST_ARGUS: Done Success
GST_ARGUS: Cleaning up
CONSUMER: Done Success
GST_ARGUS: Done Success
GST_ARGUS: Creating output stream
CONSUMER: Waiting until producer is connected...
GST_ARGUS: Available Sensor modes :
GST_ARGUS: 3280 x 2464 FR = 21.000000 fps Duration = 47619048 ; Analog Gain range min 1.000000, max 10.625000; Exposure Range min 13000, max 683709000;

GST_ARGUS: 3280 x 1848 FR = 28.000001 fps Duration = 35714284 ; Analog Gain range min 1.000000, max 10.625000; Exposure Range min 13000, max 683709000;

GST_ARGUS: 1920 x 1080 FR = 29.999999 fps Duration = 33333334 ; Analog Gain range min 1.000000, max 10.625000; Exposure Range min 13000, max 683709000;

GST_ARGUS: 1640 x 1232 FR = 29.999999 fps Duration = 33333334 ; Analog Gain range min 1.000000, max 10.625000; Exposure Range min 13000, max 683709000;

GST_ARGUS: 1280 x 720 FR = 59.999999 fps Duration = 16666667 ; Analog Gain range min 1.000000, max 10.625000; Exposure Range min 13000, max 683709000;



[ WARN:0] global ./modules/videoio/src/cap_gstreamer.cpp (1100) open OpenCV | GStreamer warning: Cannot query video position: status=0, value=-1, duration=-1


GST_ARGUS: Cleaning up
CONSUMER: Done Success
GST_ARGUS: Done Success


[ WARN:0] global ./modules/videoio/src/cap_gstreamer.cpp (1390) setProperty OpenCV | GStreamer warning: GStreamer: unhandled property


GST_ARGUS: Creating output stream
CONSUMER: Waiting until producer is connected...
GST_ARGUS: Available Sensor modes :
GST_ARGUS: 3280 x 2464 FR = 21.000000 fps Duration = 47619048 ; Analog Gain range min 1.000000, max 10.625000; Exposure Range min 13000, max 683709000;

GST_ARGUS: 3280 x 1848 FR = 28.000001 fps Duration = 35714284 ; Analog Gain range min 1.000000, max 10.625000; Exposure Range min 13000, max 683709000;

GST_ARGUS: 1920 x 1080 FR = 29.999999 fps Duration = 33333334 ; Analog Gain range min 1.000000, max 10.625000; Exposure Range min 13000, max 683709000;

GST_ARGUS: 1640 x 1232 FR = 29.999999 fps Duration = 33333334 ; Analog Gain range min 1.000000, max 10.625000; Exposure Range min 13000, max 683709000;

GST_ARGUS: 1280 x 720 FR = 59.999999 fps Duration = 16666667 ; Analog Gain range min 1.000000, max 10.625000; Exposure Range min 13000, max 683709000;

GST_ARGUS: Running with following settings:
   Camera index = 0 
   Camera mode  = 4 
   Output Stream W = 1280 H = 7

[ WARN:0] global ./modules/videoio/src/cap_gstreamer.cpp (1100) open OpenCV | GStreamer warning: Cannot query video position: status=0, value=-1, duration=-1


GST_ARGUS: Cleaning up
CONSUMER: Done Success
GST_ARGUS: Done Success
GST_ARGUS: Creating output stream


[ WARN:0] global ./modules/videoio/src/cap_gstreamer.cpp (1390) setProperty OpenCV | GStreamer warning: GStreamer: unhandled property


CONSUMER: Waiting until producer is connected...
GST_ARGUS: Available Sensor modes :
GST_ARGUS: 3280 x 2464 FR = 21.000000 fps Duration = 47619048 ; Analog Gain range min 1.000000, max 10.625000; Exposure Range min 13000, max 683709000;

GST_ARGUS: 3280 x 1848 FR = 28.000001 fps Duration = 35714284 ; Analog Gain range min 1.000000, max 10.625000; Exposure Range min 13000, max 683709000;

GST_ARGUS: 1920 x 1080 FR = 29.999999 fps Duration = 33333334 ; Analog Gain range min 1.000000, max 10.625000; Exposure Range min 13000, max 683709000;

GST_ARGUS: 1640 x 1232 FR = 29.999999 fps Duration = 33333334 ; Analog Gain range min 1.000000, max 10.625000; Exposure Range min 13000, max 683709000;

GST_ARGUS: 1280 x 720 FR = 59.999999 fps Duration = 16666667 ; Analog Gain range min 1.000000, max 10.625000; Exposure Range min 13000, max 683709000;

GST_ARGUS: Running with following settings:
   Camera index = 1 
   Camera mode  = 4 
   Output Stream W = 1280 H = 720 
   seconds to Run    = 0 
   F

### Recording (세션마다 재실행)

In [23]:
# --- Recording loop ---
# 카메라 스레드는 그대로 두고 state만 리셋
state.update(speed=0.0, steer=0.0, recording=False)

# 시작 직전에 latest 한 번 비워서 fresh frame부터
latest["L"] = None
latest["R"] = None

# fresh frame 들어올 때까지 대기
t_wait = time.time()
while (latest["L"] is None or latest["R"] is None) and time.time() - t_wait < 2.0:
    time.sleep(0.02)

session_dir = DATA_ROOT / f"{SESSION_NAME}_{time.strftime('%Y%m%d_%H%M%S')}"
(session_dir / "images").mkdir(parents=True, exist_ok=True)
ann_path = session_dir / "annotation.txt"
ann = open(ann_path, "w")
ann.write("# filename xpos ypos segment steer_tel speed_tel\n")

dt = 1.0 / FPS
saved = 0

def record_loop():
    global saved
    try:
        while not state["stop"]:
            t0 = time.time()
            L = latest["L"]
            R_ = latest["R"]
            if L is None or R_ is None:
                time.sleep(0.005); continue

            lr = cv2.remap(L, m1x, m1y, cv2.INTER_LINEAR)
            rx, ry, rw, rh = ROI
            lr_roi = lr[ry:ry + rh, rx:rx + rw]

            if state["recording"] and state["segment"] != "pause":
                ts = time.time_ns() // 1_000_000
                fname = f"{ts}.jpg"
                cv2.imwrite(
                    str(session_dir / "images" / fname),
                    cv2.cvtColor(lr_roi, cv2.COLOR_RGB2BGR),
                    [cv2.IMWRITE_JPEG_QUALITY, 90],
                )
                ann.write(f"{fname} -1 -1 {state['segment']} "
                          f"{state['steer']:.3f} {state['speed']:.3f}\n")
                ann.flush()
                saved += 1

            if hasattr(motor, "base_speed_ctrl"):
                thr = state["speed"]
                L_m = max(-0.5, min(0.5, thr * (1 - state["steer"])))
                R_m = max(-0.5, min(0.5, thr * (1 + state["steer"])))
                motor.base_speed_ctrl(L_m, R_m)

            slack = dt - (time.time() - t0)
            if slack > 0: time.sleep(slack)
    finally:
        if hasattr(motor, "base_speed_ctrl"): motor.base_speed_ctrl(0, 0)
        ann.close()
        print(f"\nsession: {session_dir}\nframes saved: {saved}")

threading.Thread(target=record_loop, daemon=True).start()
print("loop started — use buttons above to control (no preview)")

loop started — use buttons above to control (no preview)


In [ ]:
# --- Cleanup ---
state["stop"] = True
time.sleep(0.5)
for name in ["cam_l", "cam_r"]:
    cam = globals().get(name)
    if cam is not None:
        try: cam.stop()
        except: pass
        try: cam._cam.cap.release()
        except: pass
print("done")

## Part B — Label

Click the road-center point on each image. `xpos, ypos` get written back to `annotation.txt`.

In [ ]:
%matplotlib widget
import matplotlib.pyplot as plt

SESSION_DIR = session_dir   # or: Path.home() / "rover_data/session01_YYYYMMDD_HHMMSS"
ANN = SESSION_DIR / "annotation.txt"

header_lines, rows = [], []
for line in open(ANN):
    s = line.strip()
    if not s or s.startswith("#"):
        header_lines.append(line.rstrip("\n"))
    else:
        rows.append(s.split())
print(f"{len(rows)} rows to label")

In [ ]:
idx = [0]
fig, ax = plt.subplots(figsize=(9, 5))

def show(i):
    ax.clear()
    if i >= len(rows):
        ax.set_title("DONE — run the next cell to save")
        fig.canvas.draw(); return
    fname = rows[i][0]
    img = cv2.imread(str(SESSION_DIR / "images" / fname))
    if img is None:
        ax.set_title(f"MISSING {fname} — skipping")
        idx[0] += 1; show(idx[0]); return
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    xp, yp = rows[i][1], rows[i][2]
    if xp != "-1":
        ax.plot(int(xp), int(yp), "r+", markersize=20, mew=2)
    ax.set_title(f"{i + 1}/{len(rows)}  {fname}  seg={rows[i][3]}  "
                 f"steer={rows[i][4]}  click road center")
    ax.set_xticks([]); ax.set_yticks([])
    fig.canvas.draw()

def onclick(event):
    if event.inaxes != ax or event.xdata is None: return
    i = idx[0]
    if i >= len(rows): return
    rows[i][1] = str(int(event.xdata))
    rows[i][2] = str(int(event.ydata))
    idx[0] += 1
    show(idx[0])

def onkey(event):
    if event.key == "b" and idx[0] > 0:
        idx[0] -= 1
        rows[idx[0]][1] = "-1"; rows[idx[0]][2] = "-1"
    elif event.key == "n":
        idx[0] += 1
    show(idx[0])

fig.canvas.mpl_connect("button_press_event", onclick)
fig.canvas.mpl_connect("key_press_event",    onkey)
show(0)
print("click: label  |  n: skip  |  b: back+unlabel")

In [ ]:
# --- Save labeled annotation ---
with open(ANN, "w") as f:
    for h in header_lines: f.write(h + "\n")
    for r in rows:         f.write(" ".join(r) + "\n")
labeled = sum(1 for r in rows if r[1] != "-1")
print(f"saved: {labeled} / {len(rows)} frames labeled in {ANN}")